<a href="https://colab.research.google.com/github/Rafid13iit/machine_learning_class_activities/blob/master/text2code_seq2seq.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# ================================================================
# TEXT-TO-PYTHON CODE GENERATION USING SEQ2SEQ MODELS
# Complete Implementation for Google Colab
# ================================================================

# ================================================================
# 1. INSTALL DEPENDENCIES
# ================================================================

!pip install datasets transformers nltk matplotlib seaborn torch torchtext -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 53.2 MB/s eta 0:00:00


In [4]:
# ================================================================
# 2. IMPORT LIBRARIES
# ================================================================

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import random
import time
import math
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

from datasets import load_dataset
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# Download NLTK data
nltk.download('punkt', quiet=True)

# Set random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [5]:
# ================================================================
# 3. LOAD DATASET
# ================================================================

print("Loading CodeSearchNet Python dataset...")
dataset = load_dataset("Nan-Do/code-search-net-python", split='train')

# Sample configuration
MAX_SAMPLES = 10000
MAX_DOCSTRING_LENGTH = 50
MAX_CODE_LENGTH = 80

# Sample the dataset
print(f"Sampling {MAX_SAMPLES} examples from dataset...")
dataset = dataset.shuffle(seed=SEED).select(range(min(MAX_SAMPLES, len(dataset))))

# Split into train/val/test
train_size = int(0.8 * len(dataset))
val_size = int(0.1 * len(dataset))
test_size = len(dataset) - train_size - val_size

train_data = dataset.select(range(train_size))
val_data = dataset.select(range(train_size, train_size + val_size))
test_data = dataset.select(range(train_size + val_size, len(dataset)))

print(f"Train samples: {len(train_data)}")
print(f"Validation samples: {len(val_data)}")
print(f"Test samples: {len(test_data)}")

# Display sample
print("\n=== Sample Example ===")
sample = train_data[0]
print(f"Docstring: {sample['docstring']}")
print(f"Code: {sample['code']}")

Loading CodeSearchNet Python dataset...


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00004-ee77a7de79eb2a(…):   0%|          | 0.00/155M [00:00<?, ?B/s]

data/train-00001-of-00004-648b3bede2edf6(…):   0%|          | 0.00/139M [00:00<?, ?B/s]

data/train-00002-of-00004-1dfd72b171e6b2(…):   0%|          | 0.00/153M [00:00<?, ?B/s]

data/train-00003-of-00004-184ab6d0e3c690(…):   0%|          | 0.00/151M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/455243 [00:00<?, ? examples/s]

Sampling 10000 examples from dataset...
Train samples: 8000
Validation samples: 1000
Test samples: 1000

=== Sample Example ===
Docstring: Deletes a stored procedure.

        :param str sproc_link:
            The link to the stored procedure.
        :param dict options:
            The request options for the request.

        :return:
            The deleted Stored Procedure.
        :rtype:
            dict
Code: def DeleteStoredProcedure(self, sproc_link, options=None):
        """Deletes a stored procedure.

        :param str sproc_link:
            The link to the stored procedure.
        :param dict options:
            The request options for the request.

        :return:
            The deleted Stored Procedure.
        :rtype:
            dict

        """
        if options is None:
            options = {}

        path = base.GetPathFromLink(sproc_link)
        sproc_id = base.GetResourceIdOrFullNameFromLink(sproc_link)
        return self.DeleteResource(path,
       

In [6]:
# ================================================================
# 4. PREPROCESSING AND TOKENIZATION
# ================================================================

class Tokenizer:
    """Simple whitespace tokenizer"""

    def tokenize(self, text):
        """Tokenize text by whitespace"""
        return text.lower().split()

tokenizer = Tokenizer()

def preprocess_data(data, max_doc_len, max_code_len):
    """Preprocess dataset"""
    processed = []

    for item in tqdm(data, desc="Preprocessing"):
        doc = item['docstring']
        code = item['code']

        if not doc or not code:
            continue

        # Tokenize
        doc_tokens = tokenizer.tokenize(doc)
        code_tokens = tokenizer.tokenize(code)

        # Filter by length
        if len(doc_tokens) <= max_doc_len and len(code_tokens) <= max_code_len and len(doc_tokens) > 0 and len(code_tokens) > 0:
            processed.append({
                'docstring_tokens': doc_tokens,
                'code_tokens': code_tokens
            })

    return processed

print("\nPreprocessing datasets...")
train_processed = preprocess_data(train_data, MAX_DOCSTRING_LENGTH, MAX_CODE_LENGTH)
val_processed = preprocess_data(val_data, MAX_DOCSTRING_LENGTH, MAX_CODE_LENGTH)
test_processed = preprocess_data(test_data, MAX_DOCSTRING_LENGTH, MAX_CODE_LENGTH)

print(f"Processed train samples: {len(train_processed)}")
print(f"Processed validation samples: {len(val_processed)}")
print(f"Processed test samples: {len(test_processed)}")


Preprocessing datasets...


Preprocessing: 100%|██████████| 1000/1000 [00:00<00:00, 3419.55it/s]

Processed train samples: 4751
Processed validation samples: 598
Processed test samples: 593


In [7]:
# ================================================================
# 5. VOCABULARY CONSTRUCTION
# ================================================================

class Vocabulary:
    """Vocabulary class for token-to-index mapping"""

    def __init__(self):
        self.token2idx = {}
        self.idx2token = {}
        self.token_counts = Counter()

        # Special tokens
        self.PAD_TOKEN = '<pad>'
        self.SOS_TOKEN = '<sos>'
        self.EOS_TOKEN = '<eos>'
        self.UNK_TOKEN = '<unk>'

        # Initialize with special tokens
        self.add_token(self.PAD_TOKEN)
        self.add_token(self.SOS_TOKEN)
        self.add_token(self.EOS_TOKEN)
        self.add_token(self.UNK_TOKEN)

        self.pad_idx = self.token2idx[self.PAD_TOKEN]
        self.sos_idx = self.token2idx[self.SOS_TOKEN]
        self.eos_idx = self.token2idx[self.EOS_TOKEN]
        self.unk_idx = self.token2idx[self.UNK_TOKEN]

    def add_token(self, token):
        """Add token to vocabulary"""
        if token not in self.token2idx:
            idx = len(self.token2idx)
            self.token2idx[token] = idx
            self.idx2token[idx] = token
        self.token_counts[token] += 1

    def build_vocabulary(self, data, min_freq=1):
        """Build vocabulary from data"""
        for item in data:
            for token in item:
                self.token_counts[token] += 1

        # Add tokens that meet minimum frequency
        for token, count in self.token_counts.items():
            if count >= min_freq and token not in self.token2idx:
                self.add_token(token)

    def tokens_to_indices(self, tokens):
        """Convert tokens to indices"""
        return [self.token2idx.get(token, self.unk_idx) for token in tokens]

    def indices_to_tokens(self, indices):
        """Convert indices to tokens"""
        return [self.idx2token.get(idx, self.UNK_TOKEN) for idx in indices]

    def __len__(self):
        return len(self.token2idx)

# Build vocabularies
print("\nBuilding vocabularies...")

src_vocab = Vocabulary()
tgt_vocab = Vocabulary()

# Extract all tokens
all_doc_tokens = [item['docstring_tokens'] for item in train_processed]
all_code_tokens = [item['code_tokens'] for item in train_processed]

# Build vocabularies
src_vocab.build_vocabulary(all_doc_tokens, min_freq=1)
tgt_vocab.build_vocabulary(all_code_tokens, min_freq=1)

print(f"Source vocabulary size: {len(src_vocab)}")
print(f"Target vocabulary size: {len(tgt_vocab)}")


Building vocabularies...
Source vocabulary size: 14517
Target vocabulary size: 60812


In [8]:
# ================================================================
# 6. DATASET CLASS
# ================================================================

class CodeDataset(Dataset):
    """PyTorch Dataset for code generation"""

    def __init__(self, data, src_vocab, tgt_vocab):
        self.data = data
        self.src_vocab = src_vocab
        self.tgt_vocab = tgt_vocab

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        # Convert to indices
        src_indices = self.src_vocab.tokens_to_indices(item['docstring_tokens'])
        tgt_indices = self.tgt_vocab.tokens_to_indices(item['code_tokens'])

        # Add SOS and EOS tokens to target
        tgt_indices = [self.tgt_vocab.sos_idx] + tgt_indices + [self.tgt_vocab.eos_idx]

        return {
            'src': torch.tensor(src_indices, dtype=torch.long),
            'tgt': torch.tensor(tgt_indices, dtype=torch.long)
        }

def collate_fn(batch):
    """Collate function for DataLoader with padding"""
    src_batch = [item['src'] for item in batch]
    tgt_batch = [item['tgt'] for item in batch]

    # Pad sequences
    src_padded = pad_sequence(src_batch, batch_first=True, padding_value=src_vocab.pad_idx)
    tgt_padded = pad_sequence(tgt_batch, batch_first=True, padding_value=tgt_vocab.pad_idx)

    return {
        'src': src_padded,
        'tgt': tgt_padded
    }

# Create datasets
train_dataset = CodeDataset(train_processed, src_vocab, tgt_vocab)
val_dataset = CodeDataset(val_processed, src_vocab, tgt_vocab)
test_dataset = CodeDataset(test_processed, src_vocab, tgt_vocab)

# Create dataloaders
BATCH_SIZE = 32

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print(f"\nCreated DataLoaders with batch size: {BATCH_SIZE}")


Created DataLoaders with batch size: 32


In [9]:
# ================================================================
# 7. MODEL IMPLEMENTATIONS
# ================================================================

# -------------------- MODEL 1: VANILLA RNN --------------------

class VanillaRNNEncoder(nn.Module):
    """Vanilla RNN Encoder"""

    def __init__(self, vocab_size, embedding_dim, hidden_dim, pad_idx):
        super(VanillaRNNEncoder, self).__init__()
        self.hidden_dim = hidden_dim

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.rnn = nn.RNN(embedding_dim, hidden_dim, batch_first=True)

    def forward(self, src):
        # src: [batch_size, src_len]
        embedded = self.embedding(src)  # [batch_size, src_len, embedding_dim]
        outputs, hidden = self.rnn(embedded)  # outputs: [batch_size, src_len, hidden_dim]
                                               # hidden: [1, batch_size, hidden_dim]
        return outputs, hidden

class VanillaRNNDecoder(nn.Module):
    """Vanilla RNN Decoder"""

    def __init__(self, vocab_size, embedding_dim, hidden_dim, pad_idx):
        super(VanillaRNNDecoder, self).__init__()
        self.vocab_size = vocab_size
        self.hidden_dim = hidden_dim

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.rnn = nn.RNN(embedding_dim, hidden_dim, batch_first=True)
        self.fc_out = nn.Linear(hidden_dim, vocab_size)

    def forward(self, input, hidden):
        # input: [batch_size, 1]
        # hidden: [1, batch_size, hidden_dim]

        embedded = self.embedding(input)  # [batch_size, 1, embedding_dim]
        output, hidden = self.rnn(embedded, hidden)  # output: [batch_size, 1, hidden_dim]
        prediction = self.fc_out(output.squeeze(1))  # [batch_size, vocab_size]

        return prediction, hidden

class VanillaRNNSeq2Seq(nn.Module):
    """Vanilla RNN Seq2Seq Model"""

    def __init__(self, encoder, decoder, device):
        super(VanillaRNNSeq2Seq, self).__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, tgt, teacher_forcing_ratio=0.5):
        # src: [batch_size, src_len]
        # tgt: [batch_size, tgt_len]

        batch_size = src.shape[0]
        tgt_len = tgt.shape[1]
        tgt_vocab_size = self.decoder.vocab_size

        # Tensor to store decoder outputs
        outputs = torch.zeros(batch_size, tgt_len, tgt_vocab_size).to(self.device)

        # Encode source sequence
        encoder_outputs, hidden = self.encoder(src)

        # First input to decoder is <sos> token
        input = tgt[:, 0].unsqueeze(1)  # [batch_size, 1]

        for t in range(1, tgt_len):
            # Decode
            output, hidden = self.decoder(input, hidden)
            outputs[:, t, :] = output

            # Teacher forcing
            teacher_force = random.random() < teacher_forcing_ratio
            top1 = output.argmax(1).unsqueeze(1)
            input = tgt[:, t].unsqueeze(1) if teacher_force else top1

        return outputs

# -------------------- MODEL 2: LSTM --------------------

class LSTMEncoder(nn.Module):
    """LSTM Encoder"""

    def __init__(self, vocab_size, embedding_dim, hidden_dim, pad_idx):
        super(LSTMEncoder, self).__init__()
        self.hidden_dim = hidden_dim

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)

    def forward(self, src):
        # src: [batch_size, src_len]
        embedded = self.embedding(src)  # [batch_size, src_len, embedding_dim]
        outputs, (hidden, cell) = self.lstm(embedded)
        # outputs: [batch_size, src_len, hidden_dim]
        # hidden: [1, batch_size, hidden_dim]
        # cell: [1, batch_size, hidden_dim]
        return outputs, (hidden, cell)

class LSTMDecoder(nn.Module):
    """LSTM Decoder"""

    def __init__(self, vocab_size, embedding_dim, hidden_dim, pad_idx):
        super(LSTMDecoder, self).__init__()
        self.vocab_size = vocab_size
        self.hidden_dim = hidden_dim

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        self.fc_out = nn.Linear(hidden_dim, vocab_size)

    def forward(self, input, hidden, cell):
        # input: [batch_size, 1]
        # hidden: [1, batch_size, hidden_dim]
        # cell: [1, batch_size, hidden_dim]

        embedded = self.embedding(input)  # [batch_size, 1, embedding_dim]
        output, (hidden, cell) = self.lstm(embedded, (hidden, cell))
        # output: [batch_size, 1, hidden_dim]

        prediction = self.fc_out(output.squeeze(1))  # [batch_size, vocab_size]

        return prediction, hidden, cell

class LSTMSeq2Seq(nn.Module):
    """LSTM Seq2Seq Model"""

    def __init__(self, encoder, decoder, device):
        super(LSTMSeq2Seq, self).__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, tgt, teacher_forcing_ratio=0.5):
        # src: [batch_size, src_len]
        # tgt: [batch_size, tgt_len]

        batch_size = src.shape[0]
        tgt_len = tgt.shape[1]
        tgt_vocab_size = self.decoder.vocab_size

        # Tensor to store decoder outputs
        outputs = torch.zeros(batch_size, tgt_len, tgt_vocab_size).to(self.device)

        # Encode source sequence
        encoder_outputs, (hidden, cell) = self.encoder(src)

        # First input to decoder is <sos> token
        input = tgt[:, 0].unsqueeze(1)  # [batch_size, 1]

        for t in range(1, tgt_len):
            # Decode
            output, hidden, cell = self.decoder(input, hidden, cell)
            outputs[:, t, :] = output

            # Teacher forcing
            teacher_force = random.random() < teacher_forcing_ratio
            top1 = output.argmax(1).unsqueeze(1)
            input = tgt[:, t].unsqueeze(1) if teacher_force else top1

        return outputs

# -------------------- MODEL 3: LSTM WITH ATTENTION --------------------

class BahdanauAttention(nn.Module):
    """Bahdanau (Additive) Attention Mechanism"""

    def __init__(self, hidden_dim):
        super(BahdanauAttention, self).__init__()
        self.hidden_dim = hidden_dim

        # Attention layers
        self.attn = nn.Linear(hidden_dim * 2, hidden_dim)
        self.v = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, hidden, encoder_outputs):
        # hidden: [batch_size, hidden_dim]
        # encoder_outputs: [batch_size, src_len, hidden_dim]

        batch_size = encoder_outputs.shape[0]
        src_len = encoder_outputs.shape[1]

        # Repeat hidden state src_len times
        hidden = hidden.unsqueeze(1).repeat(1, src_len, 1)  # [batch_size, src_len, hidden_dim]

        # Calculate energy
        energy = torch.tanh(self.attn(torch.cat((hidden, encoder_outputs), dim=2)))
        # energy: [batch_size, src_len, hidden_dim]

        # Calculate attention scores
        attention = self.v(energy).squeeze(2)  # [batch_size, src_len]

        return torch.softmax(attention, dim=1)

class BidirectionalLSTMEncoder(nn.Module):
    """Bidirectional LSTM Encoder"""

    def __init__(self, vocab_size, embedding_dim, hidden_dim, pad_idx):
        super(BidirectionalLSTMEncoder, self).__init__()
        self.hidden_dim = hidden_dim

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True, bidirectional=True)

        # Linear layer to combine bidirectional hidden states
        self.fc_hidden = nn.Linear(hidden_dim * 2, hidden_dim)
        self.fc_cell = nn.Linear(hidden_dim * 2, hidden_dim)

    def forward(self, src):
        # src: [batch_size, src_len]
        embedded = self.embedding(src)  # [batch_size, src_len, embedding_dim]

        outputs, (hidden, cell) = self.lstm(embedded)
        # outputs: [batch_size, src_len, hidden_dim * 2]
        # hidden: [2, batch_size, hidden_dim]
        # cell: [2, batch_size, hidden_dim]

        # Combine forward and backward hidden states
        hidden = torch.tanh(self.fc_hidden(torch.cat((hidden[0], hidden[1]), dim=1)))
        cell = torch.tanh(self.fc_cell(torch.cat((cell[0], cell[1]), dim=1)))

        # hidden: [batch_size, hidden_dim]
        # cell: [batch_size, hidden_dim]

        # Add dimension for compatibility
        hidden = hidden.unsqueeze(0)  # [1, batch_size, hidden_dim]
        cell = cell.unsqueeze(0)  # [1, batch_size, hidden_dim]

        # Reduce outputs to hidden_dim
        outputs = self.fc_hidden(outputs)  # [batch_size, src_len, hidden_dim]

        return outputs, (hidden, cell)

class AttentionLSTMDecoder(nn.Module):
    """LSTM Decoder with Attention"""

    def __init__(self, vocab_size, embedding_dim, hidden_dim, pad_idx, attention):
        super(AttentionLSTMDecoder, self).__init__()
        self.vocab_size = vocab_size
        self.hidden_dim = hidden_dim
        self.attention = attention

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(embedding_dim + hidden_dim, hidden_dim, batch_first=True)
        self.fc_out = nn.Linear(hidden_dim, vocab_size)

    def forward(self, input, hidden, cell, encoder_outputs):
        # input: [batch_size, 1]
        # hidden: [1, batch_size, hidden_dim]
        # cell: [1, batch_size, hidden_dim]
        # encoder_outputs: [batch_size, src_len, hidden_dim]

        embedded = self.embedding(input)  # [batch_size, 1, embedding_dim]

        # Calculate attention weights
        attn_weights = self.attention(hidden.squeeze(0), encoder_outputs)
        # attn_weights: [batch_size, src_len]

        attn_weights = attn_weights.unsqueeze(1)  # [batch_size, 1, src_len]

        # Calculate context vector
        context = torch.bmm(attn_weights, encoder_outputs)
        # context: [batch_size, 1, hidden_dim]

        # Concatenate embedded input and context
        lstm_input = torch.cat((embedded, context), dim=2)
        # lstm_input: [batch_size, 1, embedding_dim + hidden_dim]

        output, (hidden, cell) = self.lstm(lstm_input, (hidden, cell))
        # output: [batch_size, 1, hidden_dim]

        prediction = self.fc_out(output.squeeze(1))  # [batch_size, vocab_size]

        return prediction, hidden, cell, attn_weights.squeeze(1)

class LSTMAttentionSeq2Seq(nn.Module):
    """LSTM Seq2Seq Model with Attention"""

    def __init__(self, encoder, decoder, device):
        super(LSTMAttentionSeq2Seq, self).__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, tgt, teacher_forcing_ratio=0.5):
        # src: [batch_size, src_len]
        # tgt: [batch_size, tgt_len]

        batch_size = src.shape[0]
        tgt_len = tgt.shape[1]
        tgt_vocab_size = self.decoder.vocab_size

        # Tensor to store decoder outputs
        outputs = torch.zeros(batch_size, tgt_len, tgt_vocab_size).to(self.device)
        attentions = torch.zeros(batch_size, tgt_len, src.shape[1]).to(self.device)

        # Encode source sequence
        encoder_outputs, (hidden, cell) = self.encoder(src)

        # First input to decoder is <sos> token
        input = tgt[:, 0].unsqueeze(1)  # [batch_size, 1]

        for t in range(1, tgt_len):
            # Decode with attention
            output, hidden, cell, attn_weights = self.decoder(input, hidden, cell, encoder_outputs)
            outputs[:, t, :] = output
            attentions[:, t, :] = attn_weights

            # Teacher forcing
            teacher_force = random.random() < teacher_forcing_ratio
            top1 = output.argmax(1).unsqueeze(1)
            input = tgt[:, t].unsqueeze(1) if teacher_force else top1

        return outputs, attentions

In [10]:
# ================================================================
# 8. INITIALIZE MODELS
# ================================================================

# Hyperparameters
EMBEDDING_DIM = 256
HIDDEN_DIM = 256
LEARNING_RATE = 0.001
TEACHER_FORCING_RATIO = 0.5
CLIP = 1.0

# Model 1: Vanilla RNN
print("\n=== Initializing Vanilla RNN Model ===")
vanilla_rnn_encoder = VanillaRNNEncoder(len(src_vocab), EMBEDDING_DIM, HIDDEN_DIM, src_vocab.pad_idx)
vanilla_rnn_decoder = VanillaRNNDecoder(len(tgt_vocab), EMBEDDING_DIM, HIDDEN_DIM, tgt_vocab.pad_idx)
vanilla_rnn_model = VanillaRNNSeq2Seq(vanilla_rnn_encoder, vanilla_rnn_decoder, device).to(device)

print(f"Vanilla RNN parameters: {sum(p.numel() for p in vanilla_rnn_model.parameters()):,}")

# Model 2: LSTM
print("\n=== Initializing LSTM Model ===")
lstm_encoder = LSTMEncoder(len(src_vocab), EMBEDDING_DIM, HIDDEN_DIM, src_vocab.pad_idx)
lstm_decoder = LSTMDecoder(len(tgt_vocab), EMBEDDING_DIM, HIDDEN_DIM, tgt_vocab.pad_idx)
lstm_model = LSTMSeq2Seq(lstm_encoder, lstm_decoder, device).to(device)

print(f"LSTM parameters: {sum(p.numel() for p in lstm_model.parameters()):,}")

# Model 3: LSTM with Attention
print("\n=== Initializing LSTM with Attention Model ===")
attention = BahdanauAttention(HIDDEN_DIM)
bidirectional_lstm_encoder = BidirectionalLSTMEncoder(len(src_vocab), EMBEDDING_DIM, HIDDEN_DIM, src_vocab.pad_idx)
attention_lstm_decoder = AttentionLSTMDecoder(len(tgt_vocab), EMBEDDING_DIM, HIDDEN_DIM, tgt_vocab.pad_idx, attention)
lstm_attention_model = LSTMAttentionSeq2Seq(bidirectional_lstm_encoder, attention_lstm_decoder, device).to(device)

print(f"LSTM with Attention parameters: {sum(p.numel() for p in lstm_attention_model.parameters()):,}")


=== Initializing Vanilla RNN Model ===
Vanilla RNN parameters: 35,176,076

=== Initializing LSTM Model ===
LSTM parameters: 35,965,580

=== Initializing LSTM with Attention Model ===
LSTM with Attention parameters: 37,148,300


In [11]:
# ================================================================
# 9. TRAINING FUNCTIONS
# ================================================================

def train_epoch(model, iterator, optimizer, criterion, clip, has_attention=False):
    """Train for one epoch"""
    model.train()
    epoch_loss = 0

    for batch in tqdm(iterator, desc="Training", leave=False):
        src = batch['src'].to(device)
        tgt = batch['tgt'].to(device)

        optimizer.zero_grad()

        # Forward pass
        if has_attention:
            output, _ = model(src, tgt, TEACHER_FORCING_RATIO)
        else:
            output = model(src, tgt, TEACHER_FORCING_RATIO)

        # Reshape for loss calculation
        output_dim = output.shape[-1]
        output = output[:, 1:].reshape(-1, output_dim)
        tgt = tgt[:, 1:].reshape(-1)

        # Calculate loss
        loss = criterion(output, tgt)

        # Backward pass
        loss.backward()

        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)

        optimizer.step()

        epoch_loss += loss.item()

    return epoch_loss / len(iterator)

def evaluate(model, iterator, criterion, has_attention=False):
    """Evaluate model"""
    model.eval()
    epoch_loss = 0

    with torch.no_grad():
        for batch in iterator:
            src = batch['src'].to(device)
            tgt = batch['tgt'].to(device)

            # Forward pass (no teacher forcing)
            if has_attention:
                output, _ = model(src, tgt, 0)
            else:
                output = model(src, tgt, 0)

            # Reshape for loss calculation
            output_dim = output.shape[-1]
            output = output[:, 1:].reshape(-1, output_dim)
            tgt = tgt[:, 1:].reshape(-1)

            # Calculate loss
            loss = criterion(output, tgt)
            epoch_loss += loss.item()

    return epoch_loss / len(iterator)

def train_model(model, train_loader, val_loader, optimizer, criterion, n_epochs, model_name, has_attention=False):
    """Complete training loop"""
    print(f"\n{'='*60}")
    print(f"Training {model_name}")
    print(f"{'='*60}")

    train_losses = []
    val_losses = []
    best_val_loss = float('inf')

    for epoch in range(n_epochs):
        start_time = time.time()

        train_loss = train_epoch(model, train_loader, optimizer, criterion, CLIP, has_attention)
        val_loss = evaluate(model, val_loader, criterion, has_attention)

        end_time = time.time()
        epoch_mins, epoch_secs = divmod(end_time - start_time, 60)

        train_losses.append(train_loss)
        val_losses.append(val_loss)

        # Save best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), f'{model_name.replace(" ", "_")}_best.pt')

        print(f'Epoch: {epoch+1:02} | Time: {int(epoch_mins)}m {int(epoch_secs)}s')
        print(f'\tTrain Loss: {train_loss:.3f} | Val Loss: {val_loss:.3f}')

    return train_losses, val_losses

In [ ]:
# ================================================================
# 10. TRAIN ALL MODELS
# ================================================================

# Loss function
criterion = nn.CrossEntropyLoss(ignore_index=tgt_vocab.pad_idx)

# Number of epochs
N_EPOCHS = 10

# Train Model 1: Vanilla RNN
vanilla_rnn_optimizer = optim.Adam(vanilla_rnn_model.parameters(), lr=LEARNING_RATE)
vanilla_rnn_train_losses, vanilla_rnn_val_losses = train_model(
    vanilla_rnn_model, train_loader, val_loader, vanilla_rnn_optimizer,
    criterion, N_EPOCHS, "Vanilla RNN", has_attention=False
)

# Train Model 2: LSTM
lstm_optimizer = optim.Adam(lstm_model.parameters(), lr=LEARNING_RATE)
lstm_train_losses, lstm_val_losses = train_model(
    lstm_model, train_loader, val_loader, lstm_optimizer,
    criterion, N_EPOCHS, "LSTM", has_attention=False
)

# Train Model 3: LSTM with Attention
lstm_attention_optimizer = optim.Adam(lstm_attention_model.parameters(), lr=LEARNING_RATE)
lstm_attention_train_losses, lstm_attention_val_losses = train_model(
    lstm_attention_model, train_loader, val_loader, lstm_attention_optimizer,
    criterion, N_EPOCHS, "LSTM with Attention", has_attention=True
)


Training Vanilla RNN


Epoch: 01 | Time: 3m 4s
	Train Loss: 8.827 | Val Loss: 8.735


Epoch: 02 | Time: 3m 1s
	Train Loss: 7.886 | Val Loss: 9.068


Epoch: 03 | Time: 3m 1s
	Train Loss: 7.611 | Val Loss: 9.225


Epoch: 04 | Time: 3m 1s
	Train Loss: 7.408 | Val Loss: 9.391


...

Epoch: 10 | Time: 3m 2s
	Train Loss: 6.445 | Val Loss: 9.690

Training LSTM


Epoch: 01 | Time: 3m 2s
	Train Loss: 8.763 | Val Loss: 8.685


Epoch: 02 | Time: 3m 1s
	Train Loss: 7.843 | Val Loss: 9.033


Epoch: 03 | Time: 3m 2s
	Train Loss: 7.591 | Val Loss: 8.990


Training:  20%|██        | 30/149 [00:35<02:21,  1.19s/it]